<!-- project-navigation -->
# 06_understanding_previous_application
Role: Exploration and data preparation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [2]:
import pandas as pd
import numpy as np

# ============================================================
# 1) Chargement de la table
# ============================================================

previous = pd.read_csv("data/raw/previous_application.csv")

print("========== DIMENSION INITIALE ==========")
print(previous.shape)

# ============================================================
# 2) Informations générales
# ============================================================

print("\n========== INFORMATIONS ==========")
print(previous.info())

# ============================================================
# 3) Colonnes disponibles
# ============================================================

print("\n========== COLONNES ==========")
print(previous.columns.tolist())

# ============================================================
# 4) Valeurs manquantes
# ============================================================

missing = previous.isnull().sum()
missing = missing[missing > 0]

missing_table = pd.DataFrame({
    "Nombre_manquant": missing,
    "Pourcentage": round((missing / len(previous)) * 100, 2)
}).sort_values(by="Pourcentage", ascending=False)

print("\n========== VALEURS MANQUANTES ==========")
display(missing_table)

# ============================================================
# 5) Analyse du statut des demandes
# ============================================================

print("\n========== NAME_CONTRACT_STATUS ==========")
print(previous["NAME_CONTRACT_STATUS"].value_counts())

# ============================================================
# 6) Traitement des valeurs manquantes AVANT agrégation
# ============================================================

# Variables financières
financial_cols = [
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_DOWN_PAYMENT",
    "CNT_PAYMENT"
]

for col in financial_cols:
    if col in previous.columns:
        previous[col] = previous[col].fillna(previous[col].median())

# ============================================================
# 7) Création des features client
# ============================================================

print("\nCréation des variables métier...")

agg_dict = {

    "SK_ID_PREV": "count",

    "AMT_APPLICATION": "mean",

    "AMT_CREDIT": "mean",

    "AMT_ANNUITY": "mean",

    "AMT_DOWN_PAYMENT": "mean",

    "CNT_PAYMENT": "mean"

}

features = previous.groupby("SK_ID_CURR").agg(agg_dict)

features.columns = [

    "NB_PREVIOUS_APPLICATIONS",

    "AVG_APPLICATION_AMOUNT",

    "AVG_CREDIT_AMOUNT",

    "AVG_ANNUITY",

    "AVG_DOWN_PAYMENT",

    "AVG_CREDIT_DURATION"

]

features = features.reset_index()

# ============================================================
# 8) Variables comportementales
# ============================================================

approved = (
    previous["NAME_CONTRACT_STATUS"] == "Approved"
).groupby(previous["SK_ID_CURR"]).sum()

refused = (
    previous["NAME_CONTRACT_STATUS"] == "Refused"
).groupby(previous["SK_ID_CURR"]).sum()

canceled = (
    previous["NAME_CONTRACT_STATUS"] == "Canceled"
).groupby(previous["SK_ID_CURR"]).sum()

unused = (
    previous["NAME_CONTRACT_STATUS"] == "Unused offer"
).groupby(previous["SK_ID_CURR"]).sum()

features = features.merge(
    approved.rename("NB_APPROVED"),
    on="SK_ID_CURR",
    how="left"
)

features = features.merge(
    refused.rename("NB_REFUSED"),
    on="SK_ID_CURR",
    how="left"
)

features = features.merge(
    canceled.rename("NB_CANCELED"),
    on="SK_ID_CURR",
    how="left"
)

features = features.merge(
    unused.rename("NB_UNUSED"),
    on="SK_ID_CURR",
    how="left"
)

# ============================================================
# 9) Ratios
# ============================================================

features["APPROVAL_RATE"] = (
    features["NB_APPROVED"] /
    features["NB_PREVIOUS_APPLICATIONS"]
)

features["REFUSAL_RATE"] = (
    features["NB_REFUSED"] /
    features["NB_PREVIOUS_APPLICATIONS"]
)

# ============================================================
# 10) Remplacement des NaN
# ============================================================

features.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_cols = features.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if col != "SK_ID_CURR":
        features[col] = features[col].fillna(0)

# ============================================================
# 11) Vérification finale
# ============================================================

print("\n========== TABLE FINALE ==========")
print(features.shape)

print("\nValeurs manquantes :")
print(features.isnull().sum().sum())

print("\nAperçu :")
display(features.head())

# ============================================================
# 12) Sauvegarde
# ============================================================

features.to_csv(
    "data/processed/previous_application_features.csv",
    index=False
)

print("\nFichier sauvegardé : previous_application_features.csv")

========== DIMENSION INITIALE ==========
(1670214, 37)

========== INFORMATIONS ==========
<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  str    
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  str    
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  str    
 11  NFLAG_LAST_APPL_IN_DAY

,Nombre_manquant,Pourcentage
RATE_INTEREST_PRIVILEGED,1664263,99.64
RATE_INTEREST_PRIMARY,1664263,99.64
AMT_DOWN_PAYMENT,895844,53.64
RATE_DOWN_PAYMENT,895844,53.64
NAME_TYPE_SUITE,820405,49.12
DAYS_LAST_DUE,673065,40.30
DAYS_FIRST_DRAWING,673065,40.30
DAYS_FIRST_DUE,673065,40.30
DAYS_TERMINATION,673065,40.30
NFLAG_INSURED_ON_APPROVAL,673065,40.30



========== NAME_CONTRACT_STATUS ==========
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

Création des variables métier...

========== TABLE FINALE ==========
(338857, 13)

Valeurs manquantes :
0

Aperçu :


,SK_ID_CURR,NB_PREVIOUS_APPLICATIONS,AVG_APPLICATION_AMOUNT,AVG_CREDIT_AMOUNT,AVG_ANNUITY,AVG_DOWN_PAYMENT,AVG_CREDIT_DURATION,NB_APPROVED,NB_REFUSED,NB_CANCELED,NB_UNUSED,APPROVAL_RATE,REFUSAL_RATE
0,100001,1,24835.50,23787.00,3951.000,2520.0,8.0,1,0,0,0,1.0,0.0
1,100002,1,179055.00,179055.00,9251.775,0.0,24.0,1,0,0,0,1.0,0.0
2,100003,3,435436.50,484191.00,56553.990,2841.0,10.0,3,0,0,0,1.0,0.0
3,100004,1,24282.00,20106.00,5357.250,4860.0,4.0,1,0,0,0,1.0,0.0
4,100005,2,22308.75,20076.75,8031.600,3051.0,12.0,1,0,1,0,0.5,0.0



Fichier sauvegardé : previous_application_features.csv


# Traitement et transformation de la table `previous_application`

## Objectif

La table `previous_application` contient l'historique des demandes de crédit effectuées par les clients auprès de Home Credit avant la demande présente.

Contrairement à `application_train`, où chaque ligne représente un client, cette table contient plusieurs lignes pour un même client (`SK_ID_CURR`), chaque ligne correspondant à une demande de crédit distincte.

L'objectif est de transformer cette table afin d'obtenir une seule ligne par client contenant des indicateurs synthétiques décrivant son historique de demandes de crédit. Ces indicateurs seront ensuite fusionnés avec `application_train` pour enrichir les données utilisées par les modèles de Machine Learning.

---

# Analyse de la table

## Dimension initiale

- Nombre de lignes : **1 670 214**
- Nombre de colonnes : **37**

Chaque ligne représente une demande de crédit effectuée par un client.

Un même client peut apparaître plusieurs fois puisqu'il peut avoir réalisé plusieurs demandes de crédit.

---

# Variables principales

Les variables les plus importantes de cette table sont notamment :

- `SK_ID_CURR` : identifiant du client.
- `SK_ID_PREV` : identifiant de la demande précédente.
- `NAME_CONTRACT_STATUS` : statut de la demande (Approved, Refused, Canceled, Unused offer).
- `AMT_APPLICATION` : montant demandé.
- `AMT_CREDIT` : montant accordé.
- `AMT_ANNUITY` : annuité du crédit.
- `AMT_DOWN_PAYMENT` : montant de l'apport initial.
- `CNT_PAYMENT` : nombre de mensualités prévues.

Ces variables permettent de décrire le comportement historique des clients vis-à-vis des demandes de crédit.

---

# Analyse des valeurs manquantes

L'analyse des données montre plusieurs variables contenant des valeurs manquantes.

Les plus importantes sont :

| Variable | Pourcentage de valeurs manquantes |
|----------|-----------------------------------:|
| RATE_INTEREST_PRIMARY | 99,64 % |
| RATE_INTEREST_PRIVILEGED | 99,64 % |
| AMT_DOWN_PAYMENT | 53,64 % |
| RATE_DOWN_PAYMENT | 53,64 % |
| NAME_TYPE_SUITE | 49,12 % |
| DAYS_FIRST_DRAWING | 40,30 % |
| DAYS_FIRST_DUE | 40,30 % |
| DAYS_LAST_DUE | 40,30 % |
| DAYS_LAST_DUE_1ST_VERSION | 40,30 % |
| DAYS_TERMINATION | 40,30 % |
| NFLAG_INSURED_ON_APPROVAL | 40,30 % |
| AMT_GOODS_PRICE | 23,08 % |
| CNT_PAYMENT | 22,29 % |
| AMT_ANNUITY | 22,29 % |

Les valeurs manquantes sont principalement dues au fait que certaines demandes de crédit ont été refusées, annulées ou n'ont jamais été finalisées.

---

# Traitement des valeurs manquantes

Avant l'agrégation des données au niveau du client, les variables numériques utilisées pour construire les indicateurs ont été traitées.

Les variables financières suivantes :

- `AMT_APPLICATION`
- `AMT_CREDIT`
- `AMT_ANNUITY`
- `AMT_DOWN_PAYMENT`
- `CNT_PAYMENT`

ont été imputées par leur **médiane**.

La médiane a été retenue car elle est moins sensible aux valeurs extrêmes que la moyenne et permet de conserver une distribution plus représentative des montants financiers.

Après la création des variables métier, les éventuelles valeurs manquantes restantes ont été remplacées par **0**, afin d'obtenir une table finale entièrement exploitable pour les étapes de modélisation.

---

# Analyse du statut des demandes

La variable `NAME_CONTRACT_STATUS` décrit l'issue de chaque demande de crédit.

La répartition observée est la suivante :

| Statut | Signification |
|---------|--------------|
| Approved | Crédit accepté |
| Refused | Crédit refusé |
| Canceled | Demande annulée |
| Unused offer | Offre non utilisée |

Cette information est particulièrement importante car elle reflète l'historique du comportement du client et constitue un indicateur pertinent du risque de défaut.

---

# Transformation au niveau du client

La table contient plusieurs demandes pour un même client.

Une agrégation par `SK_ID_CURR` est donc réalisée afin d'obtenir une seule ligne par client.

Les informations détaillées de chaque demande sont transformées en variables synthétiques représentant le comportement global du client.

---

# Variables créées

## Nombre total de demandes

Cette variable correspond au nombre total de demandes effectuées par le client.

```
NB_PREVIOUS_APPLICATIONS
```

Elle mesure l'expérience du client avec Home Credit.

---

## Nombre de crédits acceptés

```
NB_APPROVED
```

Nombre de demandes ayant été acceptées.

---

## Nombre de crédits refusés

```
NB_REFUSED
```

Nombre de demandes refusées.

---

## Nombre de demandes annulées

```
NB_CANCELED
```

Nombre de demandes annulées par le client ou l'organisme.

---

## Nombre d'offres non utilisées

```
NB_UNUSED
```

Nombre d'offres accordées mais finalement non utilisées.

---

## Montant moyen demandé

```
AVG_APPLICATION_AMOUNT
```

Moyenne des montants demandés sur l'ensemble des demandes du client.

---

## Montant moyen accordé

```
AVG_CREDIT_AMOUNT
```

Montant moyen effectivement accordé.

Cette variable permet de mesurer la capacité du client à obtenir les montants sollicités.

---

## Annuité moyenne

```
AVG_ANNUITY
```

Montant moyen des annuités des crédits précédents.

---

## Apport initial moyen

```
AVG_DOWN_PAYMENT
```

Montant moyen des apports initiaux réalisés par le client.

---

## Durée moyenne des crédits

```
AVG_CREDIT_DURATION
```

Calculée à partir de la variable `CNT_PAYMENT`.

Elle représente le nombre moyen de mensualités des crédits obtenus.

---

## Taux d'acceptation

Le taux d'acceptation est calculé par :

\[
APPROVAL\_RATE =
\frac{NB\_APPROVED}
{NB\_PREVIOUS\_APPLICATIONS}
\]

Cette variable indique la proportion de demandes acceptées parmi toutes les demandes du client.

---

## Taux de refus

Le taux de refus est calculé par :

\[
REFUSAL\_RATE =
\frac{NB\_REFUSED}
{NB\_PREVIOUS\_APPLICATIONS}
\]

Il mesure la fréquence des refus de crédit.

---

# Exemple de transformation

## Avant agrégation

| SK_ID_CURR | SK_ID_PREV | Statut | Montant |
|------------|------------|---------|---------:|
|100001|111|Approved|20000|
|100001|222|Canceled|15000|
|100001|333|Approved|30000|

Le client possède trois demandes.

---

## Après agrégation

| SK_ID_CURR | NB_PREVIOUS_APPLICATIONS | NB_APPROVED | NB_CANCELED | APPROVAL_RATE |
|------------|-------------------------:|------------:|------------:|--------------:|
|100001|3|2|1|0,67|

Toutes les informations sont désormais regroupées sur une seule ligne.

---

# Résultat obtenu

Après transformation :

- une seule ligne représente chaque client ;
- la table contient **338 857 clients** ;
- **13 variables** comportementales ont été créées ;
- aucune valeur manquante ne subsiste.

La table finale est sauvegardée sous le nom :

```
previous_application_features.csv
```

---

# Conclusion

La transformation de la table `previous_application` permet de convertir un historique détaillé de demandes de crédit en un ensemble d'indicateurs synthétiques décrivant le comportement du client.

Ces nouvelles variables constituent des informations pertinentes pour la prédiction du risque de défaut et seront fusionnées avec les autres tables du projet avant la phase de modélisation.